# climagrid: train, compare, and export stress forecasters (Kaggle)

This notebook trains the asset stress forecaster on **three history lengths**
(10 years, 15 years, and the full ~24-year NASA POWER record), backtests all
three on the **same recent test period**, picks the most accurate, and saves
every trained model for download.

> **Honest framing.** We forecast environmental **stress** (IEEE C57.91
> transformer heat-aging), not equipment failure. See the project's
> Validation Notes. A forecast of rising stress is a lead-time aid for
> scheduling inspections.

**Design choices that make this efficient and fair:**
- The full history is fetched from NASA POWER **once** and cached, then the
  10/15-year sets are slices of it (no refetching).
- All three windows are scored on the **same recent test fold**, so the only
  thing that differs is how much training history each model saw, a fair
  accuracy comparison.
- Models are saved to `/kaggle/working` (downloadable). Inference later needs
  only the most recent ~30 days of history per asset, not the full record.

## Setup

On Kaggle: turn **Internet ON** in the notebook settings (Settings -> Internet),
so the NASA POWER fetch works. The forecasting code is not on PyPI yet, so we
install it from the feature branch.

In [ ]:
%pip install -q "climagrid[ml] @ git+https://github.com/TemidireAdesiji/climagrid@feat/forecasting"

In [ ]:
import json
import os
import warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

import climagrid
from climagrid.forecasting import ForecastConfig, evaluate
from climagrid.forecasting.dataset import build_supervised_frame, build_training_panel
from climagrid.forecasting.models import LightGBMForecaster

warnings.filterwarnings("ignore")

ON_KAGGLE = Path("/kaggle/working").exists()
OUT = Path("/kaggle/working") if ON_KAGGLE else Path("climagrid_out")
CACHE = OUT / "cache"
CACHE.mkdir(parents=True, exist_ok=True)
print("climagrid", climagrid.__version__, "| output dir:", OUT, "| on kaggle:", ON_KAGGLE)

## 1. Assets and configuration

Point `ASSETS_CSV` at your own asset list (`asset_id, lat, lon`). If you do
not have one, the cell falls back to the bundled 33-substation sample.

In [ ]:
ASSETS_CSV = "your_assets.csv"  # <-- set to your own CSV (asset_id, lat, lon)

if not os.path.exists(ASSETS_CSV):
    local = Path(climagrid.__file__).resolve().parents[2] / "examples" / "data" / "sample_assets.csv"
    if local.exists():
        ASSETS_CSV = str(local)
    else:
        url = "https://raw.githubusercontent.com/TemidireAdesiji/climagrid/main/examples/data/sample_assets.csv"
        ASSETS_CSV = str(OUT / "sample_assets.csv")
        pd.read_csv(url).to_csv(ASSETS_CSV, index=False)

assets_df = pd.read_csv(ASSETS_CSV, dtype={"asset_id": str})
print(len(assets_df), "assets from", ASSETS_CSV)
assets_df.head()

In [ ]:
config = ForecastConfig(
    targets=["feat_thermal_aging_factor"],
    horizon_days=7,
    lags=[1, 2, 3, 7, 14, 30],
    rolling_windows=[7, 30],
    quantiles=[0.1, 0.5, 0.9],
    cache_dir=CACHE,
)
TARGET = config.targets[0]

# NASA POWER hourly data begins 2001-01-01. Fixed end date for reproducibility.
START_FULL = datetime(2001, 1, 1, tzinfo=timezone.utc)
END = datetime(2025, 12, 31, tzinfo=timezone.utc)
print("inference needs the most recent", config.min_inference_history_days, "days of history per asset")

## 2. Fetch the full history once (cached)

This is the slow step: one NASA POWER call per asset for the full record
(roughly 0.8 MB per asset-year over the wire). It is cached to parquet, so
re-running the notebook skips the download. Expect this to take a while for
the full 33-asset, ~24-year pull.

In [ ]:
panel = build_training_panel(ASSETS_CSV, START_FULL, END, config)
if panel.empty:
    raise RuntimeError(
        "Empty panel: no data was fetched. On Kaggle, enable Internet "
        "(Settings -> Internet); also confirm NASA POWER is reachable."
    )
panel_path = OUT / "daily_panel_full.parquet"
panel.to_parquet(panel_path, index=False)
print("full panel:", panel.shape, "|", panel["date"].min().date(), "->", panel["date"].max().date())
print("saved ->", panel_path)
panel.head()

## 3. Train and backtest each history window

For each of 10 years, 15 years, and the full record we slice the cached panel,
run a rolling-origin backtest (training LightGBM per fold), and train a final
model on the whole slice for saving. The backtest test folds sit at the end of
the timeline, so every window is scored on the same recent period.

In [ ]:
def subset_last_years(frame: pd.DataFrame, years: int | None) -> pd.DataFrame:
    if years is None:
        return frame
    cutoff = frame["date"].max() - pd.DateOffset(years=years)
    return frame[frame["date"] >= cutoff]

WINDOWS = [("10yr", 10), ("15yr", 15), ("full", None)]
all_scores = []
models = {}

for label, years in WINDOWS:
    sub = subset_last_years(panel, years)
    span = (sub["date"].max() - sub["date"].min()).days / 365.25
    scores = evaluate(sub, config, n_splits=3, test_size_days=90)
    # Fit, then conformally calibrate the intervals on a held-out final year.
    sup_sub = build_supervised_frame(sub, TARGET, config)
    _dts = sorted(sup_sub["date"].unique())
    _calib_start = pd.Timestamp(_dts[-365])
    _train_cut = _calib_start - pd.Timedelta(days=config.effective_embargo_days)
    model = (
        LightGBMForecaster(config)
        .fit(sup_sub[sup_sub["date"] < _train_cut], TARGET)
        .calibrate(sup_sub[sup_sub["date"] >= _calib_start], TARGET)
    )
    model_path = OUT / f"model_{label}.joblib"
    model.save(model_path)
    models[label] = model
    all_scores.append(scores.assign(window=label, train_years=round(span, 1)))
    print(f"{label:5s}: ~{span:4.1f} yr, {len(sub):7d} rows -> {model_path.name}")

all_scores = pd.concat(all_scores, ignore_index=True)

## 4. Compare and pick the most accurate

The most accurate model is the one with the lowest mean out-of-sample MAE on
the shared test period. We also show skill against the persistence and
climatology baselines and the 80% interval coverage (target 0.80).

In [ ]:
comparison = (
    all_scores.groupby("window")
    .agg(
        train_years=("train_years", "first"),
        mae=("mae", "mean"),
        rmse=("rmse", "mean"),
        skill_vs_persistence=("skill_vs_persistence", "mean"),
        skill_vs_climatology=("skill_vs_climatology", "mean"),
        interval_coverage=("interval_coverage", "mean"),
    )
    .round(4)
    .sort_values("mae")
)
best = comparison.index[0]
print("Most accurate (lowest mean MAE):", best)
comparison

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for label, _ in WINDOWS:
    by_h = all_scores[all_scores["window"] == label].groupby("horizon_day")["skill_vs_persistence"].mean()
    ax.plot(by_h.index, by_h.values, marker="o", label=label)
ax.axhline(0.0, color="grey", ls="--", lw=0.8)
ax.set_xlabel("Forecast horizon (days)")
ax.set_ylabel("Skill vs persistence")
ax.set_title("Forecast skill by training-history length")
ax.legend()
fig.tight_layout()
plt.show()

## 5. Save everything for download

On Kaggle, files written to `/kaggle/working` are downloadable from the
notebook's Output tab (and can be saved as a versioned Kaggle Dataset). We
write all three models, the cached panel, the metrics, and a manifest.

In [ ]:
comparison.to_csv(OUT / "model_comparison.csv")
all_scores.to_csv(OUT / "backtest_scores.csv", index=False)
manifest = {
    "best_window": best,
    "best_model_file": f"model_{best}.joblib",
    "all_model_files": [f"model_{label}.joblib" for label, _ in WINDOWS],
    "panel_file": panel_path.name,
    "target": TARGET,
    "horizon_days": config.horizon_days,
    "min_inference_history_days": config.min_inference_history_days,
}
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2))

print("Saved to", OUT, ":")
for path in sorted(OUT.glob("*")):
    if path.is_file():
        print(f"  {path.name:28s} {path.stat().st_size / 1024:8.0f} KB")

## 6. Inference uses only the recent ~30 days

A saved model does **not** need the full training history to forecast. The
predictors are autoregressive lags and trailing rolling windows that reach
back at most `config.min_inference_history_days`. Below we reload the winning
model and forecast from only the last ~30 days per asset, the realistic
production pattern (in deployment you would fetch just this recent window).

In [ ]:
lookback = config.min_inference_history_days + 10  # small buffer
recent = panel.sort_values(["asset_id", "date"]).groupby("asset_id", group_keys=False).tail(lookback)
print(f"using the last {lookback} days per asset (full record has up to",
      int(panel.groupby('asset_id').size().max()), "days)")

loaded = LightGBMForecaster.load(OUT / f"model_{best}.joblib")
sup_recent = build_supervised_frame(recent, TARGET, config)
latest = sup_recent.sort_values("date").groupby("asset_id", as_index=False).tail(1)
forecast = loaded.predict(latest, TARGET)
forecast.head(config.horizon_days)

In [ ]:
asset = forecast["asset_id"].iloc[0]
one = forecast[forecast["asset_id"] == asset].sort_values("horizon_day")
fig, ax = plt.subplots(figsize=(7, 4))
ax.fill_between(one["forecast_date"], one["p10"], one["p90"], alpha=0.25, label="p10-p90")
ax.plot(one["forecast_date"], one["p50"], marker="o", label="p50 (median)")
ax.set_title(f"'{best}' model forecast from ~{lookback} recent days: {asset}")
ax.set_xlabel("Forecast date")
ax.set_ylabel(TARGET)
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

## Takeaways

- We trained on 10, 15, and the full ~24 years, compared them on the same
  recent test period, and let the data pick the most accurate, rather than
  assuming more history is always better.
- All models are saved under `/kaggle/working` for download; `manifest.json`
  records which one won.
- Inference needs only the recent ~30 days per asset, so the saved model is
  cheap to serve.
- This forecasts environmental stress for inspection lead time. It is still
  not a failure prediction; combine it with your own failure records to go
  further.